# DS340 Final Project Scaffold: Combine Drill Sequences -> NFL WR Route Performance

**Question.** Does a learned representation of a WR's Combine drill movement (LSTM encoders + pooling) predict NFL route performance (e.g. separation at pass forward) *beyond* the official Combine numbers and draft position?

**Pipeline.**
```
reps in each drill family -> [LSTM encoder(s)] -> one vector per rep
        -> [pooling: mean | max | learned-weight | self-attention] -> player vector
        -> (+ optional attributes) -> [MLP head] -> one prediction per route family
```

**Feature sets compared (same downstream evaluation for all):**
- **A**: official Combine numbers + height/weight (+ draft pick as a variation)
- **B**: hand-built kinematics from the tracking data (top speed, braking, turn sharpness, ...)
- **C**: learned (LSTM / attention model)

**How the sections map to your milestones**
- Sections 1-7 (setup, audit, cleaning, tensors, targets, baselines A/B) = your **mid-project TA milestone** (aim for ~Nov 17).
- Sections 8-11 = the neural model and grouped cross-validation.
- Sections 12-13 = variations/ablations and interpretation (figures for the paper + lightning talk).

**Conventions.**
- Anything marked `VERIFY` is a column name or value I inferred from public descriptions of the data. Check it against your actual files.
- Anything marked `TODO` is yours to complete.
- **AI statement (course requirement):** this scaffold was generated with Claude. The syllabus requires you to disclose AI use and comment AI-written code. Keep the `# [AI-SCAFFOLD]` tags on lines you did not write yourselves, and delete the tag on code you rewrite. Update your written AI statement to match what you actually kept.

---
**Decisions locked in so far (from the data audit; revisit only with evidence)**
- **Population:** WRs only, using premade `wr_only_*.csv` files (106 WRs have Combine tracking).
- **Target:** average `separation_at_pass_forward` per route family (`vertical`, `break`, `short`), adjusted for route type. FLAT and SCREEN are excluded.
- **Min routes per family:** `MIN_ROUTES = 20` (split-half reliability plateaus there; 15 and 25 are sensitivity checks). Note: this is a **count of routes**, not a distance.
- **Combine drills in the encoder:** the WR route drills (after merging name variants and mirroring left-side reps) plus the 40-yard dash. Left out: 3-cone and shuttle tracking (too sparse), ball tracking (about a third of attempts), `GAUNTLET_DRILL`, `OVER_SHOULDER_ADJUST`, and `CIRCUS` (2025 class only).
- **Channels:** `speed`, `acc_tan` (signed, along the path), `acc_cent` (signed, sideways). The sensor's `a` is the total acceleration magnitude, so it adds no new information. Raw channels are kept as an ablation.

## 1. Setup and configuration

Keep every tunable value here so the ablations in section 12 are just different configs. Everything that other sections treat as a decision (target definition, route families, drill scope, channel sets) is defined **once** in this section. Do not redefine `ROUTE_FAMILY`, `ROUTE_FAMS` or `SEP_COL` in later cells.

The course runs on Colab, so install anything missing in a separate cell; torch, sklearn, pandas and scipy are normally preinstalled.

In [ ]:
import random, math, itertools, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.stats import spearmanr

SEED = 0
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Folder holding the WR-only CSVs (and, for the one-time build, the full competition CSVs).
# Your earlier notebook read/wrote them in the working directory, hence ".".
# On Kaggle: Path("/kaggle/input/nfl-big-data-bowl-2027"); on Colab: a mounted Drive folder.
DATA_DIR = Path(".")

In [ ]:
# ---------------- Target definition (decided from the audit) ----------------
SEP_COL = "separation_at_pass_forward"

MIN_ROUTES = 20        # min routes WITH a separation value per family to get a target (a COUNT, not a distance).
                       # Split-half reliability plateaus around 20; run 15 and 25 as sensitivity checks.
MIN_ROUTE_REPS = 3     # min route-drill reps per WR (run 6 as a sensitivity check)

# Single source of truth for route families.
# FLAT and SCREEN are left out on purpose: they happen at or behind the line, so separation
# at pass forward means something different there.
ROUTE_FAMILY = {
    "GO": "vertical", "POST": "vertical", "CORNER": "vertical", "WHEEL": "vertical",
    "OUT": "break", "IN": "break", "CROSS": "break", "SLANT": "break", "ANGLE": "break",
    "HITCH": "short",
}
ROUTE_FAMS = sorted(set(ROUTE_FAMILY.values()))

# ---------------- Combine drill scope ----------------
# drill_type -> encoder family. 3-cone / shuttle TRACKING is too sparse (about 0.3 reps per WR):
# their official times go into feature set A instead.
FAMILY_OF_DRILL_TYPE = {"FORTY_YARD_DASH": "sprint", "SKILL_DRILLS_WR": "route"}
FAMILIES = sorted(set(FAMILY_OF_DRILL_TYPE.values()))
FAMILY_ID = {f: i for i, f in enumerate(FAMILIES)}

# Left out of v1: ball-skill drills (not real routes) and a drill run in only one draft class (CIRCUS = 2025 only).
EXCLUDE_BASE = {"CIRCUS", "GAUNTLET_DRILL", "OVER_SHOULDER_ADJUST"}

# ---------------- Channel sets (this is the ablation in section 12) ----------------
# All of these columns are created by add_kinematics() in section 4.
CHANNEL_SETS = {
    "derived":   ["speed", "acc_tan", "acc_cent"],            # default: signed, physically meaningful
    "derived+a": ["speed", "acc_tan", "acc_cent", "a"],       # plus the sensor's total acceleration
    "turn":      ["speed", "acc_tan", "turn"],                # turn rate instead of sideways acceleration
    "raw":       ["s", "a", "dir_sin", "dir_cos"],            # "as given" (dir is NOT mirrored for left-side drills)
}

CFG = dict(
    T = 100,                              # fixed length every rep is resampled to
    channels = CHANNEL_SETS["derived"],
    min_routes_per_family = MIN_ROUTES,   # used by build_targets() in section 6
    min_reps_per_player = MIN_ROUTE_REPS, # used by make_player_table() in section 7
    hidden = 32,                  # LSTM hidden size (keep small: about 100 players!)
    drill_dim = 4,                # size of the drill embedding
    pool = "mean",                # "mean" | "max" | "weighted" | "attention"
    per_family_encoders = False,  # False = one shared LSTM; True = one LSTM per drill family
    use_attrs = True,             # feed official measurables into the head
    dropout = 0.3,
    lr = 1e-3, weight_decay = 1e-2,
    epochs = 200, patience = 20, batch_size = 16,
    n_folds = 5, n_seeds = 3,
)
print(DEVICE, "| channels:", CFG["channels"])

## 2. Load the data (WR-only files)

Everything downstream uses WR-only tables, so the big competition CSVs are needed only once. The cell below **builds** `wr_only_players.csv`, `wr_only_results.csv`, `wr_only_trk.csv` and `wr_only_play.csv` from the full files if they do not exist yet, and otherwise just loads them. Delete the WR-only files to force a rebuild.

Notes:
- WRs are defined by `nfl_position == "WR"` in `players.csv`. About 106 of them have Combine tracking.
- Combine tracking has `entity_type` `PLAYER` and `BALL` rows. Ball rows carry the player's `nfl_id` (they survive the WR filter), so ball frames can be joined on `(event_id, nfl_id)`. They cover only about a third of attempts, so v1 does not use them.
- `time` is parsed to a datetime once, here.
- **Do not load the `game_tracking_*.csv` files.** They are huge, and you do not need them: separation is already in `player_play.csv`.

In [ ]:
WR_FILES = {k: DATA_DIR / f"wr_only_{k}.csv" for k in ["players", "results", "trk", "play"]}

def build_wr_files():
    """One-time: needs the FULL competition CSVs in DATA_DIR. Skipped when the WR-only files already exist."""
    players_full = pd.read_csv(DATA_DIR / "players.csv")
    ids = set(players_full.loc[players_full["nfl_position"] == "WR", "nfl_id"])
    sources = {"players": "players.csv", "results": "combine_results.csv",
               "trk": "combine_tracking.csv", "play": "player_play.csv"}
    for key, fname in sources.items():
        df = pd.read_csv(DATA_DIR / fname)
        df[df["nfl_id"].isin(ids)].to_csv(WR_FILES[key], index=False)
        print("wrote", WR_FILES[key])

if not all(p.exists() for p in WR_FILES.values()):
    build_wr_files()

wr_only_players = pd.read_csv(WR_FILES["players"])
wr_only_results = pd.read_csv(WR_FILES["results"])
wr_only_trk     = pd.read_csv(WR_FILES["trk"], parse_dates=["time"])
wr_only_play    = pd.read_csv(WR_FILES["play"])

# Later sections (5-14) use these shorter names for the same WR-only tables.
players, results, play = wr_only_players, wr_only_results, wr_only_play
wr_ids = set(wr_only_players["nfl_id"])

In [ ]:
# Sanity checks: fix any VERIFY item here before going further.
for name, df in [("players", wr_only_players), ("results", wr_only_results),
                 ("trk", wr_only_trk), ("play", wr_only_play)]:
    print(f"{name:8s} {df.shape}")

print("\nWRs in players:", len(wr_ids))
print("WRs in results:", wr_only_results["nfl_id"].nunique())
print("WRs with Combine tracking:", wr_only_trk["nfl_id"].nunique())
print("WRs with no tracking:", sorted(wr_ids - set(wr_only_trk["nfl_id"].dropna())))

print("\nentity_type:", wr_only_trk["entity_type"].value_counts(dropna=False).to_dict())
print("null nfl_id in tracking:", int(wr_only_trk["nfl_id"].isna().sum()))
print("time dtype:", wr_only_trk["time"].dtype)

need = {"play": [SEP_COL, "route_ran", "nfl_id"],
        "trk": ["event_id", "nfl_id", "entity_type", "time", "drill_type", "drill_name", "attempt",
                "draft_year", "x", "y", "s", "a", "dir"],
        "results": ["nfl_id", "draft_year", "combine_position"]}
for key, cols in need.items():
    df = {"play": wr_only_play, "trk": wr_only_trk, "results": wr_only_results}[key]
    missing = [c for c in cols if c not in df.columns]
    print(f"missing columns in {key}:", missing)

In [ ]:
# OPTIONAL: restrict to regular-season plays. Entrants warn that player_play includes preseason game ids,
# and your count of separation values is a bit above the roughly 37.5k other entrants report.
# TODO: check games.csv, then set REGULAR_SEASON_ONLY = True and fix the VERIFY names. Keep it False until then.
REGULAR_SEASON_ONLY = False

if REGULAR_SEASON_ONLY:
    games = pd.read_csv(DATA_DIR / "games.csv")
    print(games.columns.tolist())
    SEASON_TYPE_COL, REG_VALUE = "season_type", "REG"          # VERIFY both
    reg_games = set(games.loc[games[SEASON_TYPE_COL] == REG_VALUE, "game_id"])
    wr_only_play = wr_only_play[wr_only_play["game_id"].isin(reg_games)].copy()   # VERIFY game_id exists in play
    play = wr_only_play
    print("play rows after regular-season filter:", len(wr_only_play))

## 3. Data audit (findings so far, and what each one decided)

| Question | What the audit showed | Decision |
|---|---|---|
| Route-drill coverage | about 14 skill-drill reps per WR (range 8-18), roughly one rep per drill | One shared LSTM over the route drills, with a drill embedding |
| Name drift | `SPEED_OUT_LEFT` and `SPEED_OUT_ROUTE_LEFT` are one drill; `POST_CORNER` ran on both sides | Merge to a base name, mirror left-side reps (section 4) |
| Class confounds | `CIRCUS` exists only for 2025 | Exclude from v1 |
| 3-cone / shuttle | about 0.3 tracking reps per WR; official times for only about a third of WRs | No tracking encoder; official times in feature set A with missing flags |
| Ball frames | about a third of attempts, absent for several drills | Not used in v1 |
| Separation coverage | recorded on about 88% of WR routes, steady across route types | Usable as an "ability to get open" target, not a targeted-only stat |
| Target noise | split-half reliability plateaus near 20 routes per family | `MIN_ROUTES = 20` |

Cells 3.1 to 3.6 reproduce these checks with the final definitions, so re-running them after any config change updates the counts. Decision gates (adjust with the TAs):
- Fewer than about 60 usable WRs in a family: merge it into another family or drop it from the multi-output head.
- Fewer than about 20 WRs in a draft class: use pooled leave-one-class-out scoring (and K-fold as the main evaluation), not a single-class holdout.

In [ ]:
# ---- 3.0 One row per Combine drill attempt ----
KEYS = ["event_id", "nfl_id"]          # identifies one drill attempt

def parse_drill(name):
    """'POST_CORNER_ROUTE_LEFT' -> ('POST_CORNER', 'LEFT'); 'GAUNTLET_DRILL' -> ('GAUNTLET_DRILL', None)."""
    n = name.replace("_ROUTE", "")             # merges SPEED_OUT_ROUTE_LEFT / SPEED_OUT_LEFT
    side = "LEFT" if n.endswith("_LEFT") else "RIGHT" if n.endswith("_RIGHT") else None
    base = n.rsplit("_", 1)[0] if side else n
    return base, side

trk_player = wr_only_trk[wr_only_trk["entity_type"] == "PLAYER"].copy()
trk_ball   = wr_only_trk[wr_only_trk["entity_type"] == "BALL"].copy()

wr_attempts = (trk_player.groupby(KEYS)
               .agg(drill_type=("drill_type", "first"), drill_name=("drill_name", "first"),
                    draft_year=("draft_year", "first"), attempt=("attempt", "first"),
                    n_frames=("time", "count"))
               .reset_index())

_parsed = {n: parse_drill(n) for n in wr_attempts["drill_name"].unique()}
wr_attempts["base"] = wr_attempts["drill_name"].map(lambda n: _parsed[n][0])
wr_attempts["side"] = wr_attempts["drill_name"].map(lambda n: _parsed[n][1])
wr_attempts["in_scope"] = (wr_attempts["drill_type"].isin(FAMILY_OF_DRILL_TYPE)
                           & ~wr_attempts["base"].isin(EXCLUDE_BASE))

# Does the attempt have ball frames? (ball rows carry the same nfl_id, so join on KEYS)
_ball = trk_ball[KEYS].drop_duplicates().assign(has_ball=True)
wr_attempts = wr_attempts.merge(_ball, on=KEYS, how="left")
wr_attempts["has_ball"] = wr_attempts["has_ball"].eq(True)

print("attempts:", len(wr_attempts), "| in scope:", int(wr_attempts["in_scope"].sum()))

In [ ]:
# ---- 3.1 Players and reps per base drill x draft year (name variants merged) ----
print("WRs per drill")
display(wr_attempts.pivot_table(index=["drill_type", "base"], columns="draft_year",
                                values="nfl_id", aggfunc="nunique", fill_value=0))
print("Reps per drill")
display(wr_attempts.pivot_table(index=["drill_type", "base"], columns="draft_year",
                                values="event_id", aggfunc="nunique", fill_value=0))
# Check: any in-scope drill with a 0 in some draft year is confounded with draft class.

In [ ]:
# ---- 3.2 Reps per WR, and ball-frame coverage ----
reps_per_wr = wr_attempts.groupby(["nfl_id", "drill_type"]).size().unstack(fill_value=0)
display(reps_per_wr.describe())

scope_reps = wr_attempts[wr_attempts["in_scope"]].groupby("nfl_id").size()
print("in-scope reps per WR:"); print(scope_reps.describe().round(1).to_string())
print("WRs with fewer than MIN_ROUTE_REPS in-scope reps:", int((scope_reps < MIN_ROUTE_REPS).sum()))

print("\nShare of attempts with ball frames, by base drill (v1 does not use them):")
print(wr_attempts.groupby("base")["has_ball"].mean().round(2).to_string())

In [ ]:
# ---- 3.3 Separation targets per WR per route family ----
routes = wr_only_play[wr_only_play["route_ran"].notna()].copy()
routes["has_sep"] = routes[SEP_COL].notna()
routes["fam"] = routes["route_ran"].map(ROUTE_FAMILY)

print("WR route rows:", len(routes), "| non-null separation rate:", round(routes["has_sep"].mean(), 3))
print("route_ran values left out of every family:", sorted(routes.loc[routes["fam"].isna(), "route_ran"].unique()))

# Separation with route-type differences removed (a GO is not a POST), so family means compare players, not route mixes.
have = routes[routes["has_sep"] & routes["fam"].notna()].copy()
have["sep_adj"] = have[SEP_COL] - have.groupby("route_ran")[SEP_COL].transform("mean")

per = (have.groupby(["nfl_id", "fam"]).size().unstack(fill_value=0)
       .reindex(index=sorted(wr_ids), columns=ROUTE_FAMS, fill_value=0))

print("\nWRs with at least N separation values, by family:")
print(pd.DataFrame({f"N>={t}": (per >= t).sum() for t in [10, 15, 20, 25]}).to_string())
print("\nWRs by number of families meeting MIN_ROUTES:")
print((per >= MIN_ROUTES).sum(axis=1).value_counts().sort_index().to_string())

In [ ]:
# ---- 3.4 Target reliability by threshold (split-half, Spearman-Brown corrected) ----
# Reliability about 0.6 means about 60% of between-player differences in the average are stable.
# A rough ceiling on any model's correlation with the target is sqrt(reliability).
def split_half(df, fam, thr=15, n_rep=200, seed=0):
    rng = np.random.default_rng(seed)
    g = df[df["fam"] == fam].groupby("nfl_id")["sep_adj"].apply(lambda s: s.to_numpy())
    g = g[g.apply(len) >= thr]
    if len(g) < 10:
        return dict(family=fam, n_players=len(g), half_r=np.nan, reliability=np.nan)
    rs = []
    for _ in range(n_rep):
        a, b = [], []
        for vals in g:
            p = rng.permutation(len(vals)); h = len(vals) // 2
            a.append(vals[p[:h]].mean()); b.append(vals[p[h:2 * h]].mean())
        rs.append(np.corrcoef(a, b)[0, 1])
    r = float(np.mean(rs))
    return dict(family=fam, n_players=len(g), half_r=round(r, 3), reliability=round(2 * r / (1 + r), 3))

rel = pd.DataFrame([{"thr": t, **split_half(have, f, t)} for t in [5, 10, 15, 20, 25, 30, 40] for f in ROUTE_FAMS])
print("Reliability:");       display(rel.pivot(index="thr", columns="family", values="reliability"))
print("Players retained:");  display(rel.pivot(index="thr", columns="family", values="n_players"))

In [ ]:
# ---- 3.5 Intersection: WRs with usable Combine reps AND usable targets ----
def usable_table(att, per, min_routes=None, min_reps=None):
    """att: one row per attempt with nfl_id, drill_type, base. Returns (tab, usable)."""
    min_routes = MIN_ROUTES if min_routes is None else min_routes
    min_reps = MIN_ROUTE_REPS if min_reps is None else min_reps
    route_reps = (att[(att["drill_type"] == "SKILL_DRILLS_WR") & ~att["base"].isin(EXCLUDE_BASE)]
                  .groupby("nfl_id").size().rename("route_reps"))
    forty_reps = att[att["drill_type"] == "FORTY_YARD_DASH"].groupby("nfl_id").size().rename("forty_reps")
    fams = list(per.columns)
    tab = (pd.DataFrame(index=sorted(wr_ids))
           .join(wr_only_results.set_index("nfl_id")[["draft_year"]])
           .join(route_reps).join(forty_reps).join(per))
    fill_cols = ["route_reps", "forty_reps"] + fams
    tab[fill_cols] = tab[fill_cols].fillna(0)
    tab["has_reps"] = tab["route_reps"] >= min_reps
    tab["n_fams"] = (tab[fams] >= min_routes).sum(axis=1)
    usable = tab[tab["has_reps"] & (tab["n_fams"] >= 1)]
    print(f"MIN_ROUTES={min_routes}, MIN_ROUTE_REPS={min_reps}")
    print("WRs considered:           ", len(tab))
    print("with enough route reps:   ", int(tab["has_reps"].sum()))
    print("with >= 1 family target:  ", int((tab["n_fams"] >= 1).sum()))
    print("usable (both):            ", len(usable))
    print("usable per family:        ", {f: int(((tab[f] >= min_routes) & tab["has_reps"]).sum()) for f in fams})
    print("usable by draft class:    ", usable["draft_year"].value_counts().sort_index().to_dict())
    return tab, usable

tab, usable = usable_table(wr_attempts, per)
print("\nDropped WRs and why:")
display(tab[~tab.index.isin(usable.index)][["draft_year", "route_reps", "forty_reps"] + ROUTE_FAMS])
# Sensitivity: usable_table(wr_attempts, per, min_routes=15) and (..., min_routes=25, min_reps=6)

In [ ]:
# ---- 3.6 Official Combine numbers: coverage (this is feature set A) ----
official_cols = ["combine_height", "combine_weight", "hand_size", "arm_length", "wing_span",
                 "ten_yd_split", "forty", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps"]
official_cols = [c for c in official_cols if c in wr_only_results.columns]
print(f"Non-null counts over {len(wr_only_results)} WR rows:")
print(wr_only_results[official_cols].notna().sum().to_string())
# ngs_*_score columns: check how they are defined before using them. ngs_athleticism_score may be derived
# from the same Combine tracking you are modeling, which would leak into the "official numbers" baseline.

## 4. Cleaning and kinematics

What the checks showed, and what we do about it:
- **`a` is unsigned and is the total acceleration magnitude.** It correlates about 0.98 with `sqrt(acc_tan**2 + acc_cent**2)` (along-path plus sideways), and only about 0.55 with the along-path part alone. So it carries no information the two components do not. We build the **signed** components: `acc_tan` (accelerating vs braking) and `acc_cent` (cutting left vs right).
- **`dir` is an angle that wraps at 360.** Never feed raw degrees. We use the wrap-safe change in heading (`turn`, deg/s), and `sin`/`cos` for the "raw" ablation.
- **`dis` is not used** (it reportedly under-counts).
- **Absolute `x`, `y` are not used** (they encode where a drill was run, not how the player moved).
- **Heading is unreliable at low speed**, so `turn` and `acc_cent` are zeroed below 1 yd/s.
- **Attempt QC:** drop attempts that are too short, have time gaps, impossible speeds or NaNs. Look at the distributions first, then set the thresholds.
- **Name variants and sides:** merge to a base drill name and mirror left-side reps so a left cut and a right cut look alike.

In [ ]:
# ---- 4.1 Kinematic channels ----
def add_kinematics(df):
    """Adds speed, acc_tan (signed, along the path), turn (deg/s), acc_cent (signed, sideways), dir_sin, dir_cos."""
    d = df.sort_values(KEYS + ["time"]).copy()
    g = d.groupby(KEYS)
    d["dt_raw"] = g["time"].diff().dt.total_seconds()                    # NaN on each attempt's first frame
    d["dt"] = d["dt_raw"].fillna(0.1).clip(lower=1e-3)
    d["speed"] = d["s"]                                                  # find_segment() in section 5 expects "speed"
    d["s_sm"] = g["s"].transform(lambda x: x.rolling(3, center=True, min_periods=1).mean())
    g = d.groupby(KEYS)                                                  # rebuild so it sees the new column
    d["acc_tan"] = g["s_sm"].diff() / d["dt"]
    d["turn"] = ((g["dir"].diff() + 180) % 360 - 180) / d["dt"]          # deg/s, wrap-safe
    # First frame of each attempt has no previous frame (clips often start in motion): copy the next frame's value.
    g = d.groupby(KEYS)
    d[["acc_tan", "turn"]] = g[["acc_tan", "turn"]].bfill().fillna(0.0)
    d.loc[d["s"] < 1.0, "turn"] = 0.0                                    # heading is unreliable at low speed
    d["acc_cent"] = d["s"] * np.radians(d["turn"])                       # signed sideways acceleration
    rad = np.radians(d["dir"])
    d["dir_sin"], d["dir_cos"] = np.sin(rad), np.cos(rad)
    return d

trk_k = add_kinematics(trk_player)
print(trk_k[["speed", "acc_tan", "turn", "acc_cent"]].describe().round(2).to_string())

In [ ]:
# ---- 4.2 Check (optional): does `a` equal total acceleration? ----
print("a has negatives:", bool((trk_k["a"] < 0).any()), "(False means unsigned)")
m = (trk_k["s"] >= 1.0) & trk_k["dt_raw"].notna()     # skip each attempt's first frame (no previous frame)
acc_total = np.hypot(trk_k["acc_tan"], trk_k["acc_cent"])
print("corr(|acc_tan|, a):", round(np.corrcoef(trk_k.loc[m, "acc_tan"].abs(), trk_k.loc[m, "a"])[0, 1], 3))
print("corr(acc_total, a):", round(np.corrcoef(acc_total[m], trk_k.loc[m, "a"])[0, 1], 3))
# Expect about 0.55 vs about 0.98 (what you observed). If the second number drops a lot, the new
# add_kinematics is not reproducing the sensor: check dt and the turn computation.

In [ ]:
# ---- 4.3 Attempt QC ----
# Step 1: look at the distributions per drill type. Step 2: set the thresholds so they only catch clear glitches.
QC = dict(min_frames=15,       # at 10 Hz, 15 frames = 1.5 s: shorter than any real drill  (TODO: tune from the table)
          max_gap_s=0.5,       # larger time gap inside an attempt = dropped frames
          max_speed=12.0)      # yd/s; the fastest 40-yard dashes are about 10-11 yd/s  (TODO: tune)

def attempt_qc(d, qc=QC):
    a = (d.groupby(KEYS)
         .agg(drill_type=("drill_type", "first"), n_frames=("time", "count"), peak_speed=("s", "max"),
              max_gap=("dt_raw", "max"), n_nan=("s", lambda x: int(x.isna().sum())))
         .reset_index())
    a["bad_short"] = a["n_frames"] < qc["min_frames"]
    a["bad_gap"]   = a["max_gap"] > qc["max_gap_s"]
    a["bad_speed"] = a["peak_speed"] > qc["max_speed"]
    a["bad_nan"]   = a["n_nan"] > 0
    a["bad"] = a[["bad_short", "bad_gap", "bad_speed", "bad_nan"]].any(axis=1)
    return a

qc_tbl = attempt_qc(trk_k)
print("Distributions by drill type:")
display(qc_tbl.groupby("drill_type")[["n_frames", "peak_speed", "max_gap"]]
        .quantile([0.0, 0.01, 0.5, 0.99, 1.0]).round(2))
print("Flags:", qc_tbl[["bad_short", "bad_gap", "bad_speed", "bad_nan", "bad"]].sum().to_dict())
print("Attempts flagged by drill type:", qc_tbl.groupby("drill_type")["bad"].sum().to_dict())

bad_keys = qc_tbl.loc[qc_tbl["bad"], KEYS].assign(_bad=True)
d_clean = trk_k.merge(bad_keys, on=KEYS, how="left")
d_clean = d_clean[d_clean["_bad"].isna()].drop(columns="_bad").copy()
print("attempts kept:", d_clean.groupby(KEYS).ngroups, "of", len(qc_tbl))

In [ ]:
# ---- 4.4 Canonical drill names, left/right mirroring, drill scope ----
_names = {n: parse_drill(n) for n in d_clean["drill_name"].unique()}
d_clean["drill_raw"]  = d_clean["drill_name"]
d_clean["drill_base"] = d_clean["drill_name"].map(lambda n: _names[n][0])
d_clean["side"]       = d_clean["drill_name"].map(lambda n: _names[n][1])

# Sign convention check: for each attempt, the direction of its hardest cut (sign of acc_cent at peak |acc_cent|).
# Only drills run on BOTH sides can verify this (POST_CORNER). Before mirroring, LEFT and RIGHT reps should have
# OPPOSITE mean signs (about -1 vs +1). After mirroring they should match.
_idx = d_clean.assign(_abs=d_clean["acc_cent"].abs()).groupby(KEYS)["_abs"].idxmax()
peak = d_clean.loc[_idx, KEYS + ["drill_base", "side", "acc_cent"]].copy()
peak["sign_before"] = np.sign(peak["acc_cent"])
peak["sign_after"] = np.where(peak["side"] == "LEFT", -peak["sign_before"], peak["sign_before"])
chk = (peak[peak["side"].notna()].groupby(["drill_base", "side"])[["sign_before", "sign_after"]]
       .mean().round(2))
display(chk)

MIRROR_LEFT = True      # set False if the check above shows LEFT and RIGHT already have matching signs
if MIRROR_LEFT:
    left = d_clean["side"] == "LEFT"
    d_clean.loc[left, ["acc_cent", "turn"]] *= -1
    # note: dir_sin / dir_cos (the "raw" channel set) are not mirrored

# Scope: drill types and drills used in v1. drill_name becomes the canonical base name,
# because section 5 builds DRILL_IDS from wr_trk["drill_name"].
in_scope = d_clean["drill_type"].isin(FAMILY_OF_DRILL_TYPE) & ~d_clean["drill_base"].isin(EXCLUDE_BASE)
wr_trk = d_clean[in_scope].copy()
wr_trk["drill_name"] = wr_trk["drill_base"]

print("\nwr_trk:", len(wr_trk), "frames |", wr_trk.groupby(KEYS).ngroups, "attempts |",
      wr_trk["nfl_id"].nunique(), "WRs")
print(wr_trk.groupby("drill_name")[KEYS[0]].nunique().to_string())

In [ ]:
# ---- 4.5 Final check: usable WRs after cleaning (compare with 3.5) ----
att_clean = (wr_trk.groupby(KEYS)
             .agg(drill_type=("drill_type", "first"), base=("drill_base", "first"))
             .reset_index())
tab_clean, usable_clean = usable_table(att_clean, per)
print("\nWRs lost to cleaning:", sorted(set(usable.index) - set(usable_clean.index)))

## 5. Turn each rep into a fixed-size tensor

Each rep becomes an array of shape `(T, C)`:
1. **Trim to the movement**: start at the first frame where speed exceeds a small threshold (the clip may start in motion, so also handle the case where it already is).
2. **Pick alignment events per family**, then resample the segment(s) between events to fixed length `T`:
   - `sprint`: motion start -> end of the run. (Entrants report the first ~10 yd of the 40 are unreliable in tracking; consider ignoring official-time comparisons.)
   - `cod` (3-cone, shuttle): motion start -> end. Optionally align on each direction change.
   - `route`: motion start -> break point (largest heading change) -> catch/end. If ball frames exist, use the ball to find the catch.
3. **Channels**: speed, signed acceleration, turn rate (add more if useful, e.g. position relative to the start).
4. **Duration** is lost by resampling, so append it as one extra *constant* channel.
5. **Standardize channels using training-fold statistics only** (done in section 9, not here; fitting scalers on all reps leaks test information).

Output: a list of rep dicts that every later section consumes.

In [ ]:
def resample(arr, T):
    """[AI-SCAFFOLD] Linearly resample a (n, C) array to (T, C)."""
    n = len(arr)
    if n == 1:
        return np.repeat(arr, T, axis=0)
    src = np.linspace(0.0, 1.0, n)
    dst = np.linspace(0.0, 1.0, T)
    return np.stack([np.interp(dst, src, arr[:, c]) for c in range(arr.shape[1])], axis=1)

def find_segment(rep_df, family):
    """TODO: return (start_idx, end_idx) of the part of the rep to keep, using family-specific events.
    Minimal version: first frame with speed > 0.5 through the last frame with speed > 0.5."""
    moving = np.where(rep_df["speed"].to_numpy() > 0.5)[0]
    if len(moving) < 5:
        return None
    return moving[0], moving[-1] + 1

# Build a stable integer id for every canonical drill name (used for the drill embedding).
DRILL_IDS = {d: i for i, d in enumerate(sorted(wr_trk["drill_name"].unique()))}

def build_reps(df, T=CFG["T"], channels=CFG["channels"]):
    """[AI-SCAFFOLD] One dict per usable rep: x (T, C+1), drill_id, family_id, nfl_id, draft_year."""
    reps = []
    for (ev, pid), rep in df.groupby(KEYS):
        dt_type = rep["drill_type"].iloc[0]
        family = FAMILY_OF_DRILL_TYPE.get(dt_type)
        if family is None:
            continue                                  # drill types we are not using
        seg = find_segment(rep, family)
        if seg is None:
            continue
        a, b = seg
        arr = rep[channels].to_numpy()[a:b]
        duration = (b - a) * 0.1
        x = resample(arr, T)
        x = np.concatenate([x, np.full((T, 1), duration)], axis=1)   # constant duration channel
        reps.append(dict(x=x.astype(np.float32), nfl_id=pid, draft_year=rep["draft_year"].iloc[0],
                         drill_id=DRILL_IDS[rep["drill_name"].iloc[0]], family_id=FAMILY_ID[family]))
    return reps

reps = build_reps(wr_trk)
print(len(reps), "reps for", len({r["nfl_id"] for r in reps}), "WRs")
# TODO: plot ~6 reps per family (speed vs resampled time) and sanity check the alignment visually.

## 6. Targets from `player_play.csv`

Goal: one number per (player, route family): average **separation at pass forward** on routes of that family.

Design decisions to make and write down:
- **Route families.** Group `route_ran` values into a few families so each has enough routes per player. The page lists GO, OUT, SLANT, CROSS, POST, CORNER, HITCH, SCREEN, FLAT, WHEEL; there may be others, so print the unmapped ones.
- **Noise control.** A player's average over 10 routes is very noisy. Use a minimum route count (`min_routes_per_family`), weight the loss by route count, and optionally **shrink** each player's mean toward the family mean (empirical Bayes).
- **Context adjustment (optional but valuable).** Separation depends on coverage (man/zone) and cushion at the snap. Fit a simple regression of separation on route type + coverage + cushion and use the *residual*; otherwise a WR who mostly faces zone looks "better".
- **Mask.** If a player has no qualifying routes in a family, that target is missing: store `ym = 0` so the loss ignores it.
- Only use **regular-season** rows if the table includes preseason ids (VERIFY how to filter).

In [ ]:
# ROUTE_FAMILY, ROUTE_FAMS and SEP_COL are defined once in section 1.

def build_targets(play_df, wr_ids):
    """[AI-SCAFFOLD] Returns dict nfl_id -> (y, ym, w), each of length K = len(ROUTE_FAMS)."""
    p = play_df[play_df["nfl_id"].isin(wr_ids)].dropna(subset=["route_ran", SEP_COL]).copy()
    p["fam"] = p["route_ran"].map(ROUTE_FAMILY)
    unmapped = p.loc[p["fam"].isna(), "route_ran"].unique()
    if len(unmapped):
        print("Routes left out of every family (expected: FLAT, SCREEN):", list(unmapped))
    p = p.dropna(subset=["fam"])
    # TODO (optional): replace SEP_COL by a residual after adjusting for coverage and cushion.
    out = {}
    for pid, g in p.groupby("nfl_id"):
        y = np.zeros(len(ROUTE_FAMS), dtype=np.float32)
        ym = np.zeros(len(ROUTE_FAMS), dtype=np.float32)
        w = np.zeros(len(ROUTE_FAMS), dtype=np.float32)
        for k, fam in enumerate(ROUTE_FAMS):
            gg = g[g["fam"] == fam]
            if len(gg) >= CFG["min_routes_per_family"]:
                y[k], ym[k], w[k] = gg[SEP_COL].mean(), 1.0, math.sqrt(len(gg))   # TODO: shrinkage
        if ym.sum() > 0:
            out[pid] = (y, ym, w)
    return out

# wr_ids comes from section 2
targets = build_targets(play, wr_ids)
print(len(targets), "WRs with at least one qualifying route family")

## 7. Baselines A and B with the evaluation harness (your milestone)

Everything here is scored with the **same folds and metrics** later used for the neural model, so numbers are comparable.

**Splits.** Players are the unit. Default: shuffled K-fold over players (each player appears in exactly one test fold). Secondary check: **leave one draft class out** (the realistic "predict the next class" test), if the audit shows enough WRs per class.

**Metrics.** Per route family: Spearman correlation between predicted and actual (ranking is what a scout cares about) and R^2. Report the mean across families and the per-family values, with a bootstrap over players for confidence intervals (n is small, so intervals matter).

**Baselines.**
- **A**: official numbers (40, 3-cone, shuttle, vertical, broad jump, height, weight; draft pick as a variation). Ridge per route family.
- **B**: hand-built kinematics per player: top speed, time to 90% of top speed, peak acceleration, hardest deceleration, peak turn rate, speed retained after the break. Ridge (or a small random forest from class) per route family.

If A or B matches the neural model, that is a result worth reporting.

In [ ]:
def make_player_table(reps, targets, results):
    """[AI-SCAFFOLD] One entry per WR that has targets and >= min_reps_per_player reps."""
    by_player = {}
    for r in reps:
        by_player.setdefault(r["nfl_id"], []).append(r)
    ids = [pid for pid in targets
           if len(by_player.get(pid, [])) >= CFG["min_reps_per_player"]]
    return ids, by_player

player_ids, reps_by_player = make_player_table(reps, targets, results)
Y  = np.stack([targets[p][0] for p in player_ids])
YM = np.stack([targets[p][1] for p in player_ids])
W  = np.stack([targets[p][2] for p in player_ids])
draft_year = results.set_index("nfl_id").loc[player_ids, "draft_year"].to_numpy()   # VERIFY

def get_splits(n, mode="kfold", seed=SEED):
    """Yield (train_idx, test_idx). mode: 'kfold' or 'draft_class'."""
    if mode == "kfold":
        yield from KFold(CFG["n_folds"], shuffle=True, random_state=seed).split(np.arange(n))
    else:
        for yr in np.unique(draft_year):
            test = np.where(draft_year == yr)[0]
            train = np.where(draft_year != yr)[0]
            yield train, test

def score(pred, Y, YM, W=None):
    """Spearman and R^2 per route family, using only players with a target for that family."""
    rows = []
    for k, fam in enumerate(ROUTE_FAMS):
        m = YM[:, k] > 0
        if m.sum() < 5:
            continue
        rho = spearmanr(pred[m, k], Y[m, k]).correlation
        ss_res = ((Y[m, k] - pred[m, k]) ** 2).sum()
        ss_tot = ((Y[m, k] - Y[m, k].mean()) ** 2).sum()
        rows.append(dict(family=fam, n=int(m.sum()), spearman=rho, r2=1 - ss_res / ss_tot))
    return pd.DataFrame(rows)

def ridge_oof(X, Y, YM, mode="kfold", alpha=10.0):
    """Out-of-fold predictions from per-family ridge. X: (n_players, n_features)."""
    oof = np.full_like(Y, np.nan)
    for tr, te in get_splits(len(X), mode):
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-8        # scale with TRAIN stats only
        for k in range(Y.shape[1]):
            trk_ = tr[YM[tr, k] > 0]
            if len(trk_) < 10:
                continue
            m = Ridge(alpha=alpha).fit((X[trk_] - mu) / sd, Y[trk_, k])
            oof[te, k] = m.predict((X[te] - mu) / sd)
    return oof

# --- Feature set A ---------------------------------------------------------------------------
# TODO: choose official columns from combine_results (VERIFY names) and build X_A, shape (n_players, n_features).
# Handle opt-outs / missing values (e.g. impute with TRAIN medians inside each fold, plus a missing flag).
# X_A = results.set_index("nfl_id").loc[player_ids, [...]].to_numpy()

# --- Feature set B ---------------------------------------------------------------------------
# TODO: per rep, compute the kinematic summaries, then take the per-player mean (or best rep).
# X_B = ...

# --- Run and record ---------------------------------------------------------------------------
# resA = score(ridge_oof(X_A, Y, YM), Y, YM); print(resA)
# resB = score(ridge_oof(X_B, Y, YM), Y, YM); print(resB)
# TODO: also run A+draft pick, and the "draft pick only" baseline.

## 8. Dataset and batching (variable number of reps per player)

A training example is a **player**: a set of reps (each `(T, C)`), their drill and family ids, an attribute vector, and the targets with their mask and weights.

Players have different numbers of reps, so the collate function pads to the largest count in the batch and returns a boolean `mask` marking real reps. Every later module must respect that mask (padding must never influence pooling or attention).

Standardization of channels and attributes uses statistics computed from the **training fold only** and is passed into the dataset.

In [ ]:
def fit_scalers(train_player_ids, reps_by_player, attrs):
    """[AI-SCAFFOLD] Channel and attribute mean/std from TRAIN players only."""
    allx = np.concatenate([r["x"] for p in train_player_ids for r in reps_by_player[p]], axis=0)  # (N*T, C)
    ch_mean, ch_std = allx.mean(0), allx.std(0) + 1e-6
    return ch_mean, ch_std, attrs.mean(0), attrs.std(0) + 1e-6

class PlayerDataset(Dataset):
    def __init__(self, idx, player_ids, reps_by_player, attrs, Y, YM, W, scalers):
        self.idx, self.player_ids = idx, player_ids
        self.reps_by_player, self.attrs = reps_by_player, attrs
        self.Y, self.YM, self.W = Y, YM, W
        self.ch_mean, self.ch_std, self.a_mean, self.a_std = scalers
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        reps = self.reps_by_player[self.player_ids[j]]
        x = np.stack([(r["x"] - self.ch_mean) / self.ch_std for r in reps])      # (R, T, C)
        return dict(
            x=torch.tensor(x, dtype=torch.float32),
            drill=torch.tensor([r["drill_id"] for r in reps]),
            fam=torch.tensor([r["family_id"] for r in reps]),
            attrs=torch.tensor((self.attrs[j] - self.a_mean) / self.a_std, dtype=torch.float32),
            y=torch.tensor(self.Y[j]), ym=torch.tensor(self.YM[j]), w=torch.tensor(self.W[j]))

def collate(batch):
    """Pad the rep dimension to the max count in the batch and build the validity mask."""
    B = len(batch)
    R = max(b["x"].shape[0] for b in batch)
    T, C = batch[0]["x"].shape[1:]
    x = torch.zeros(B, R, T, C)
    drill = torch.zeros(B, R, dtype=torch.long)
    fam = torch.zeros(B, R, dtype=torch.long)
    mask = torch.zeros(B, R, dtype=torch.bool)
    for i, b in enumerate(batch):
        r = b["x"].shape[0]
        x[i, :r], drill[i, :r], fam[i, :r], mask[i, :r] = b["x"], b["drill"], b["fam"], True
    out = dict(x=x, drill=drill, fam=fam, mask=mask)
    for k in ["attrs", "y", "ym", "w"]:
        out[k] = torch.stack([b[k] for b in batch])
    return out

## 9. Model

Three parts, each a variation point:

1. **RepEncoder**: an LSTM over one rep. The drill embedding is concatenated to *every timestep*, so a shared LSTM can still behave differently per drill. With `per_family_encoders=True` each drill family gets its own LSTM (bias-variance tradeoff: more specialization, less data per model).
2. **Pooling** over the reps of a player, respecting the mask: `mean`, `max`, `weighted` (one learned score per rep, softmax) or `attention` (a single small self-attention block, then mean over valid reps).
3. **Head**: an MLP that outputs one value per route family (multi-output), with optional attributes concatenated to the player vector.

Gotchas:
- A player with **zero** valid reps would give NaNs in attention (all keys masked). Section 7 already drops players below `min_reps_per_player`.
- Keep the model tiny. About 100 players cannot support a large network.

In [ ]:
class RepEncoder(nn.Module):
    """[AI-SCAFFOLD] LSTM over one rep; drill embedding appended at every timestep."""
    def __init__(self, n_channels, n_drills, drill_dim, hidden, dropout):
        super().__init__()
        self.drill_emb = nn.Embedding(n_drills, drill_dim)
        self.lstm = nn.LSTM(n_channels + drill_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, drill_id):                      # x: (N, T, C), drill_id: (N,)
        d = self.drill_emb(drill_id).unsqueeze(1).expand(-1, x.size(1), -1)
        _, (h, _) = self.lstm(torch.cat([x, d], dim=-1))
        return self.drop(h[-1])                          # (N, hidden)

class Pool(nn.Module):
    """[AI-SCAFFOLD] Mask-aware pooling over reps. h: (B, R, D), mask: (B, R) True = real rep."""
    def __init__(self, kind, dim, heads=2, dropout=0.1):
        super().__init__()
        self.kind = kind
        self.last_weights = None                         # saved for interpretation (section 13)
        if kind == "weighted":
            self.score = nn.Linear(dim, 1)
        if kind == "attention":
            self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(dim)
    def forward(self, h, mask):
        m = mask.unsqueeze(-1).float()
        if self.kind == "mean":
            return (h * m).sum(1) / m.sum(1).clamp(min=1)
        if self.kind == "max":
            return h.masked_fill(~mask.unsqueeze(-1), -1e9).max(1).values
        if self.kind == "weighted":
            s = self.score(h).squeeze(-1).masked_fill(~mask, -1e9)
            w = torch.softmax(s, dim=1)
            self.last_weights = w.detach()
            return (w.unsqueeze(-1) * h).sum(1)
        if self.kind == "attention":
            out, w = self.mha(h, h, h, key_padding_mask=~mask, need_weights=True)
            self.last_weights = w.detach()               # (B, R, R); average over queries to rank reps
            h2 = self.norm(h + out)                      # residual + norm
            return (h2 * m).sum(1) / m.sum(1).clamp(min=1)
        raise ValueError(self.kind)

class PlayerModel(nn.Module):
    def __init__(self, n_channels, n_drills, n_attrs, n_out, cfg):
        super().__init__()
        self.cfg = cfg
        n_enc = len(FAMILIES) if cfg["per_family_encoders"] else 1
        self.encoders = nn.ModuleDict({str(i): RepEncoder(n_channels, n_drills, cfg["drill_dim"],
                                                          cfg["hidden"], cfg["dropout"])
                                       for i in range(n_enc)})
        self.pool = Pool(cfg["pool"], cfg["hidden"])
        head_in = cfg["hidden"] + (n_attrs if cfg["use_attrs"] else 0)
        self.head = nn.Sequential(nn.Linear(head_in, 32), nn.ReLU(), nn.Dropout(cfg["dropout"]),
                                  nn.Linear(32, n_out))
    def encode_reps(self, x, drill, fam, mask):
        B, R, T, C = x.shape
        fx, fd = x.reshape(B * R, T, C), drill.reshape(-1)
        ff = fam.reshape(-1) if self.cfg["per_family_encoders"] else torch.zeros_like(fd)
        valid = mask.reshape(-1)
        out = x.new_zeros(B * R, self.cfg["hidden"])
        for key, enc in self.encoders.items():
            sel = (ff == int(key)) & valid
            if sel.any():
                out[sel] = enc(fx[sel], fd[sel])
        return out.reshape(B, R, -1)
    def forward(self, b):
        h = self.encode_reps(b["x"], b["drill"], b["fam"], b["mask"])
        v = self.pool(h, b["mask"])                      # player vector
        if self.cfg["use_attrs"]:
            v = torch.cat([v, b["attrs"]], dim=-1)
        return self.head(v)                              # (B, n_out)

## 10. Training

- **Loss**: masked, weighted MSE. Entries with `ym = 0` (family not run / too few routes) contribute nothing; `w` upweights players with more routes (their averages are less noisy).
- **Early stopping** uses a validation set carved out of the *training* players. Never early-stop on the test fold.
- **Regularization**: dropout, AdamW weight decay, small hidden size, gradient clipping. Consider data augmentation (small random time shifts or noise on `x`) as a variation.
- Targets are in separation units; standardizing `Y` per family using train statistics usually helps optimization (TODO).

In [ ]:
def masked_mse(pred, y, ym, w):
    num = (((pred - y) ** 2) * ym * w).sum()
    den = (ym * w).sum().clamp(min=1e-8)
    return num / den

def to_dev(b):
    return {k: v.to(DEVICE) for k, v in b.items()}

@torch.no_grad()
def predict(model, dl):
    model.eval()
    return torch.cat([model(to_dev(b)).cpu() for b in dl]).numpy()

@torch.no_grad()
def eval_loss(model, dl):
    model.eval()
    tot, n = 0.0, 0
    for b in dl:
        b = to_dev(b)
        tot += masked_mse(model(b), b["y"], b["ym"], b["w"]).item() * len(b["y"])
        n += len(b["y"])
    return tot / max(n, 1)

def fit(model, train_dl, val_dl, cfg):
    """[AI-SCAFFOLD] AdamW + early stopping on the inner validation loss; restores the best weights."""
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    best, best_state, bad = float("inf"), None, 0
    for ep in range(cfg["epochs"]):
        model.train()
        for b in train_dl:
            b = to_dev(b)
            opt.zero_grad()
            loss = masked_mse(model(b), b["y"], b["ym"], b["w"])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        v = eval_loss(model, val_dl)
        if v < best - 1e-5:
            best, bad = v, 0
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return best

## 11. Grouped cross-validation for the neural model

For every outer fold:
1. Split the training players into **inner train / inner validation** (about 85/15) for early stopping.
2. Fit scalers on inner-train players only.
3. Train `n_seeds` models with different seeds and **average their predictions** (small-data networks are high variance; report the spread across seeds too).
4. Store out-of-fold predictions for the test players.

Then score the OOF matrix with the same `score()` used for the baselines. Leakage checklist: scalers, drill/route mappings that depend on data, imputation, and early stopping must never see test-fold players.

In [ ]:
def run_cv(cfg, attrs, mode="kfold"):
    """[AI-SCAFFOLD] Returns an OOF prediction matrix (n_players, n_route_families)."""
    n = len(player_ids)
    oof = np.full_like(Y, np.nan)
    n_channels = reps[0]["x"].shape[1]
    for fold, (tr, te) in enumerate(get_splits(n, mode)):
        rng = np.random.RandomState(SEED + fold)
        perm = rng.permutation(tr)
        n_val = max(1, int(0.15 * len(perm)))
        inner_val, inner_tr = perm[:n_val], perm[n_val:]

        scalers = fit_scalers([player_ids[i] for i in inner_tr], reps_by_player, attrs[inner_tr])
        mk = lambda idx: DataLoader(
            PlayerDataset(idx, player_ids, reps_by_player, attrs, Y, YM, W, scalers),
            batch_size=cfg["batch_size"], shuffle=False, collate_fn=collate)
        dl_tr = DataLoader(PlayerDataset(inner_tr, player_ids, reps_by_player, attrs, Y, YM, W, scalers),
                           batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate)
        dl_val, dl_te = mk(inner_val), mk(te)

        preds = []
        for s in range(cfg["n_seeds"]):
            set_seed(SEED + 100 * fold + s)
            model = PlayerModel(n_channels, len(DRILL_IDS), attrs.shape[1], Y.shape[1], cfg).to(DEVICE)
            fit(model, dl_tr, dl_val, cfg)
            preds.append(predict(model, dl_te))
        oof[te] = np.mean(preds, axis=0)
    return oof

# TODO: build `attrs` (n_players, n_attrs) = the same official numbers used in feature set A.
# attrs = X_A
# oof_C = run_cv(CFG, attrs)
# resC = score(oof_C, Y, YM); print(resC)
# TODO: put resA, resB, resC side by side (one table, with bootstrap CIs over players).

## 12. Variations and ablations (this is your experiment section)

Run each as a config change, using the same folds and seeds. Keep a results DataFrame (one row per experiment) and save it to disk after every run so a crashed Colab session does not cost you results.

| Variation | What it answers |
|---|---|
| `pool`: mean vs max vs weighted vs attention | Does attention over reps help, or is mean pooling enough? |
| `per_family_encoders`: False vs True | Shared LSTM vs one per drill family (bias-variance tradeoff) |
| drill embedding on/off | Does the model need to know which drill it is reading? |
| leave out one drill family at a time | Which families carry the signal? |
| `use_attrs`: with / without official numbers (and with draft pick) | Does the learned part add information beyond A? |
| `hidden`, `dropout`, `weight_decay` | Where does overfitting start? |
| rep selection: all reps vs best rep per drill | Does the model benefit from using all reps? |
| matched vs mismatched drill-route pairs | Football-facing hypothesis: does the WR route drill predict break-route separation better than the 40? |
| split mode: kfold vs draft_class | Does performance survive a realistic "next class" test? |
| LSTM vs a simple non-sequential model on summary features | Does the sequence model earn its complexity? |

Always include the baselines A and B in the same table. A null result (the neural model does not beat A or B) is a legitimate finding: report it honestly and discuss why (sample size, noise in targets, selection).

In [ ]:
EXPERIMENTS = {
    "pool_mean":       dict(pool="mean"),
    "pool_max":        dict(pool="max"),
    "pool_weighted":   dict(pool="weighted"),
    "pool_attention":  dict(pool="attention"),
    "per_family_enc":  dict(per_family_encoders=True),
    "no_attrs":        dict(use_attrs=False),
    # TODO: add drill-ablation entries (requires a way to filter reps by family before building the table)
}

all_results = []
def run_experiments(attrs, mode="kfold"):
    for name, over in EXPERIMENTS.items():
        cfg = {**CFG, **over}
        oof = run_cv(cfg, attrs, mode)
        r = score(oof, Y, YM)
        r.insert(0, "experiment", name)
        all_results.append(r)
        pd.concat(all_results).to_csv("results_so_far.csv", index=False)   # checkpoint after every run
    return pd.concat(all_results)

# results_df = run_experiments(attrs)
# summary = results_df.groupby("experiment")[["spearman", "r2"]].mean()

## 13. Interpretation (what makes this useful to a coach, not just a number)

- **Pooling weights.** For `weighted` / `attention` models, pull `model.pool.last_weights` on test players and aggregate by drill: which drills does the model lean on for each route family? Treat this as a hint, not proof. Confirm with the drill-ablation experiment.
- **Permutation importance.** Shuffle one drill family's reps across players in the test fold and measure the drop in Spearman.
- **Residual analysis.** Which WRs does the model under- and over-predict? Is it related to draft slot, route volume, or opt-outs?
- **Compare against A.** Show a scatter of A-only predictions vs C predictions for the same players. Where do they disagree, and who is right?
- **Figures to keep (they feed both your paper and the Big Data Bowl writeup):** (1) per-family Spearman for A / B / C with CIs, (2) pooling variants, (3) drill-ablation drops, (4) example rep traces with the attention weights, (5) one "scouting card" style summary.

In [ ]:
# TODO: after training a final model on all training players, run the test loader and collect pool weights.
# Example sketch:
#   model.eval(); _ = model(to_dev(batch)); w = model.pool.last_weights   # (B, R) or (B, R, R)
#   Map each rep back to its drill_id, group by drill name, and average the weight per (route family, drill).

## 14. Wrap-up checklist

**For DS340 (final paper, talk, code)**
- [ ] Say exactly which parts are lecture material (LSTMs, backprop, regularization, trees/ensembles for baselines) and which you taught yourselves (event alignment of tracking data, attention-based pooling over variable-size sets, grouped evaluation, masked losses).
- [ ] AI disclosure: matches what the code comments say (`# [AI-SCAFFOLD]` tags kept only where code is unchanged).
- [ ] The honest limits section: ~100 WRs, noisy targets, selection into the Combine, three draft classes.

**For the NFL Big Data Bowl (deadline Jan 6, 2027; check the Rules tab for the current requirements)**
- [ ] The writeup must use the Combine tracking data and explicitly link it to regular-season performance.
- [ ] Reframe results for football readers: lead with the question and the finding, show the scouting-style figure, keep the model description short.
- [ ] Publish a clean public notebook that runs top to bottom.